<a href="https://colab.research.google.com/github/mohanQAEngineer/Cross-market-analysis/blob/main/Cross_market_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Installing requests

In [ ]:
pip install requests

#1.Crypto currencies

In [ ]:
import requests
records=[]
for i in range(1,6):
    url="https://api.coingecko.com/api/v3/coins/markets?vs_currency=inr&per_page=250&order=market_cap_desc&page=1&sparkline=False"
    response=requests.get(url)
    if response.status_code==200:
        data=response.json()
        records.extend(data)
        print(f"page{i} fetched successfully.records:{len(data)}")
    else:
        print(f"failed to fetch page{i}.status code:{response.status_code}")
print(f"\ntotal records fetched:{len(records)}")

#Installing pandas

In [ ]:
pip install pandas

#Converting into a dataframe

In [ ]:
import pandas as pd

crypto_df=pd.DataFrame(records)
crypto_df

#Filter using pandas

In [ ]:
crypto_df.head()

#Retriving the crypto currency data frame

In [ ]:
crypto_df

#converting timestamp into datetime

In [ ]:
crypto_df['last_updated']=pd.to_datetime(crypto_df['last_updated']).dt.date

#Checking data types,null values of the crypto dataframe

In [ ]:
crypto_df.info()

#Converting last_updated column from object to proper date

In [ ]:
crypto_df['last_updated']=pd.to_datetime(crypto_df['last_updated'])
print(crypto_df)
print(crypto_df.dtypes)


#Display the current data of crypto_df

In [ ]:
crypto_df

#Display the length of the crypto_df

In [ ]:
print(len(crypto_df))

#2.Top 3 Coins – Historical Prices
#pick the top 3 from your dataset based on market_cap_rank
#Replace bitcoin with other coin IDs (e.g., ethereum, tether

In [ ]:

import time
coins=['bitcoin','ethereum','tether']
records=[]
for i in coins:
    url=f"https://api.coingecko.com/api/v3/coins/{i}/market_chart?vs_currency=inr&days=365"
    response=requests.get(url)

    if response.status_code==200:
       data=response.json()
       print(i, response.status_code)

       #extracting price datas
       for price in data['prices']:
        records.append({'coin_id': i,
                        'date': pd.to_datetime(price[0], unit="ms").date(),
                        'price_inr': price[1]})
        print(f"{i}data fetched successfully")

    else:
        print(f"failed to fetch{i}")
        print("Status Code:", response.status_code)
        print(response.text)

    time.sleep(15)




#Pushing it to dataframe

In [ ]:
coins_df=pd.DataFrame(records)
coins_df

#Checking data types of coins_df

In [ ]:
coins_df.info()

#Checking length of the coins_df

In [ ]:
len(coins_df)

#Checking response of the coins

In [ ]:
coins = ["bitcoin", "ethereum", "tether"]

for coin in coins:
    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart?vs_currency=inr&days=365"

    response = requests.get(url)

    print(coin, response.status_code)

#converting data type from object to date

In [ ]:
coins_df['date']=pd.to_datetime(coins_df['date'])
coins_df
coins_df.dtypes

#Retrive the coins_df

In [ ]:
coins_df

#3.Oil prices

In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv'
oil_df=pd.read_csv(url)
oil_df

#Checking the data type of oil_df

In [ ]:
oil_df.info()

#Converting the Date type into date

In [ ]:
oil_df['Date']=pd.to_datetime(oil_df['Date'])
oil_df
oil_df.info()

#Filter date between 2020 to 2026

In [ ]:
filter_df = oil_df[
    (oil_df["Date"] >= "2020-01-01") &
    (oil_df["Date"] <= "2026-12-31")
]

print(filter_df)

#Checking length of the oil_df

In [ ]:
len(oil_df)

#4.Stock Prices (Yahoo Finance API)

In [ ]:
import yfinance as yf
import pandas as pd

#choos indices

tickers=['^GSPC','^IXIC','^NSEI']
start_date='2020-01-01'
end_date='2026-03-09'

#Download historical data

stocks_df=yf.download(tickers,start=start_date,end=end_date,group_by='tickers')
stocks_df.head()

#Checking the column names of stocks_df
#Sqlite doesn't support multi index column so have to convert into single index column

In [ ]:
print(stocks_df.columns)
print(type(stocks_df.columns))

#Converting the multi index column into single index column

In [ ]:
stocks_df.columns = [
    f"{ticker}_{field}"
    for ticker, field in stocks_df.columns
]

stocks_df = stocks_df.reset_index()

#Checking the info of stocks_df whether it is modified or not(into sigle index)

In [ ]:
stocks_df.info()

#Reset the particular ticker

In [ ]:
stocks_df['^NSEI'].reset_index()

In [ ]:
stocks_df['^GSPC'].reset_index()

In [ ]:
stocks_df['^IXIC'].reset_index()

#Concating 03 stocks

In [ ]:
dfs = []

for ticker in tickers:
    df = stocks_df[ticker].copy()      # Get one ticker's data
    df.reset_index(inplace=True)       # Convert Date index to column
    df["Ticker"] = ticker              # Add ticker name
    dfs.append(df)

# Concatenate all three DataFrames
final_df = pd.concat(dfs, ignore_index=True)

print(final_df.head().reset_index)

#//end
#Created and collected 4 types of datas

#Creating database and Connecting to sqlite

In [ ]:
import sqlite3
conn=sqlite3.connect('cross_market_analysis.db')
cursor=conn.cursor()

#inserting collected datas into sqlite

In [ ]:
coins_df.to_sql('coins_data',if_exists='replace',con=conn)

In [ ]:
oil_df.to_sql('oil_data',if_exists='replace',con=conn)

In [ ]:
stocks_df.to_sql('stocks_data',conn,if_exists='replace',index=False)

#To delete current database(If needed)

In [ ]:
import os

conn = sqlite3.connect("cross_market_analysis.db")

# Perform database operations...

conn.close()      # Close the connection

os.remove("cross_market_analysis.db")

print("Database deleted.")

#crypto_df converting dic into json format for sql connection
#Sqlite doesn't support dictionary format

In [ ]:
import json

for col in crypto_df.columns:
    crypto_df[col] = crypto_df[col].apply(
        lambda x: json.dumps(x) if isinstance(x, dict) else x
    )

In [ ]:
crypto_df.to_sql('crypto_df',if_exists='replace',con=conn,index=False)

#Reading pushed datas

In [ ]:
pd.read_sql('select * from crypto_df',con=conn)

In [ ]:
pd.read_sql('select * from stocks_data',con=conn)

In [ ]:
pd.read_sql('select * from oil_data',con=conn)

In [ ]:
pd.read_sql('select * from coins_data',con=conn)

#Streamli setup

In [ ]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

#Installing streamlit option menu

In [ ]:
pip install streamlit-option_menu

#Creating and configuring the new streamlit file

In [ ]:
%%writefile crossmarketanalysis.py

import streamlit as st
import sqlite3
import pandas as pd
from streamlit_option_menu import option_menu

# Connecting to sql

conn=sqlite3.connect('cross_market_analysis.db')
cursor=conn.cursor()

st.title('CROSS MARKET ANALYSIS')
st.divider()

with st.sidebar:
    selected = option_menu(
    'Main Menu',
    ['Market Overview', 'SQL Query Runner', 'Top 03 Crypto Analysis'],
    icons=['house', 'database', 'currency-bitcoin'],
    menu_icon='cast',
    default_index=0
)
if selected=='Market Overview':
  st.title('Welcome to market overview')

  col1, col2 = st.columns(2)

  with col1:
        start_date = st.date_input("Start Date")

  with col2:
        end_date = st.date_input("End Date")

#coin avg price

  query = """
  SELECT ROUND(AVG(price_inr),2) AS bitcoin_avg
  FROM coins_data
  WHERE coin_id='bitcoin'
  AND date BETWEEN ? AND ?;
  """

  bitcoin_avg = pd.read_sql_query(
    query,
    conn,
    params=(start_date, end_date))

  btc_avg = bitcoin_avg.iloc[0]["bitcoin_avg"]

  st.write(bitcoin_avg)


#Oil average

  query = """
  SELECT ROUND(AVG(price),2) AS oil_avg
  FROM oil_data
  WHERE date BETWEEN ? AND ?;
  """

  oil_avg = pd.read_sql_query(
    query,
    conn,
    params=(start_date, end_date))

  oil_price = oil_avg.iloc[0]["oil_avg"]

  st.write(oil_avg)

#SP average

  query = """
  SELECT ROUND(AVG("^GSPC_Close"),2) AS sp500_avg
  FROM stocks_data
  WHERE Date between ? AND ?;
  """

  sp500_avg = pd.read_sql_query(
    query,
    conn,
    params=(start_date, end_date))

  sp500 = sp500_avg.iloc[0]["sp500_avg"]

  st.write(sp500_avg)

#NIFTY average

  query = """
  SELECT ROUND(AVG("^NSEI_Close"),2) AS nifty_avg
  FROM stocks_data
  WHERE Date BETWEEN ? AND ?;
  """

  nifty_avg = pd.read_sql_query(
    query,
    conn,
    params=(start_date, end_date))

  nifty = nifty_avg.iloc[0]["nifty_avg"]

  st.write(nifty_avg)

#Display snapchat

  query = """
  SELECT
    c.date,
    ROUND(c.price_inr,2) AS Bitcoin,
    ROUND(o.price,2) AS Oil,
    ROUND(s."^GSPC_Close",2) AS SP500,
    ROUND(s."^NSEI_Close",2) AS NIFTY
  FROM coins_data c
  INNER JOIN oil_data o
  ON c.date = o.date
  INNER JOIN stocks_data s
  ON c.date = s.Date
  WHERE c.coin_id='bitcoin'
  AND c.date BETWEEN ? AND ?
  ORDER BY c.date;
  """

  snapshot_df = pd.read_sql_query(
    query,
    conn,
    params=(start_date, end_date))

  st.subheader("Daily Market Snapshot")
  st.dataframe(snapshot_df, use_container_width=True)


elif selected == "SQL Query Runner":

  st.title('Welcome to sql runner')

  query_option = {
      "crypto(1).Find the top 3 cryptocurrencies by market cap":
      """SELECT id,MAX(market_cap) AS market_cap from crypto_df group by id order by market_cap desc limit 3""",

      "crypto(2).List all coins where circulating supply exceeds 90% of total supply":
      """ SELECT id,circulating_supply from crypto_df
      where circulating_supply>total_supply*0.9 group by id
      order by circulating_supply desc""",

      "crypto(3).Get coins that are within 10% of their all-time-high":
      """SELECT DISTINCT
      id,
      ROUND(current_price, 2) AS current_price,
      ROUND(ath, 2) AS all_time_high,
      ROUND((current_price / ath) * 100, 2) AS pct_of_ath
      FROM crypto_df
      WHERE ath > 0
      AND current_price >= 0.9 * ath
      ORDER BY pct_of_ath DESC;""",

      "crypto(4).Find the avg market cap rank of coins with volume above 1B":
      """SELECT id, AVG(market_cap_rank) AS avg_market_cap_rank
      FROM crypto_df
      WHERE total_volume > 1000000000 group by id order by market_cap_rank desc;""",

      "crypto(5).Get the most recently updated coin":
      """SELECT id,last_updated from crypto_df order by last_updated desc limit 1;""",

      "coins(6).Find the highest daily price of Bitcoin in the last 365 days":
      """SELECT MAX(price_inr) AS highest_price
      FROM coins_data
      WHERE coin_id = 'bitcoin'
      AND date >= DATE('now', '-1 year');""",

      "coins(7).Calculate the average daily price of Ethereum in the past 1 year":
      """SELECT AVG(price_inr) AS avg_price
      FROM coins_data
      WHERE coin_id = 'ethereum'
      AND date >= DATE('now', '-1 year');""",

      "coins(8).Show the daily price trend of Bitcoin in January 2026":
      """SELECT date, price_inr
      FROM coins_data
      WHERE coin_id = 'bitcoin'
      AND date BETWEEN '2026-01-01' AND '2026-01-31';""",

      "coins(9).Find the coin with the highest average price over 1 year":
      """SELECT coin_id, AVG(price_inr) AS avg_price
      FROM coins_data
      GROUP BY coin_id
      ORDER BY avg_price DESC
      LIMIT 1;""",

      "coins(10).Get the % change in Bitcoin’s price between Sep 2024 and Sep 2025":
      """SELECT (price_inr - LAG(price_inr) OVER (ORDER BY date)) / LAG(price_inr) OVER (ORDER BY date) * 100 AS price_change
      FROM coins_data
      WHERE coin_id = 'bitcoin'
      AND date BETWEEN '2024-09-01' AND '2025-09-30';""",

      "oil(11).Find the highest oil price in the last 5 years":
      """SELECT MAX(price) AS highest_price
      FROM oil_data
      WHERE Date >= DATE('now', '-5 years');""",

      "oil(12).Get the average oil price per year":
      """SELECT strftime('%Y', Date) AS year, AVG(price) AS avg_price
      FROM oil_data
      GROUP BY year
      ORDER BY year;""",

      "oil(13).Show oil prices during COVID crash (March–April 2020)":
      """SELECT Date, price
      FROM oil_data
      WHERE Date BETWEEN '2020-01-01' AND '2020-04-30';""",

      "oil(14).Find the lowest price of oil in the last 10 years":
      """SELECT MIN(price) AS lowest_price
      FROM oil_data
      WHERE Date >= DATE('now', '-10 years');""",

      "oil(15).Calculate the volatility of oil prices (max-min difference per year)":
      """SELECT strftime('Y', Date) AS year, MAX(price), MIN(price), MAX(price) - MIN(price) AS volatility
      FROM oil_data
      GROUP BY year
      ORDER BY year;""",

      "stocks(16).Get all stock prices for a given ticker":
      """SELECT Date, ROUND("^GSPC_Close",2) AS SP500, ROUND("^NSEI_Close",2) AS NIFTY,
      ROUND("^IXIC_Close",2) AS NASDAQ
      FROM stocks_data
      WHERE Date BETWEEN '2025-01-01' AND '2026-01-31';""",

      "stocks(17).Find the highest closing price for NASDAQ (^IXIC)":
      """SELECT MAX("^IXIC_Close") AS highest_price
      FROM stocks_data;""",

      "stocks(18).List top 5 days with highest price difference (high - low) for S&P 500 (^GSPC)":
      """SELECT Date, ROUND("^GSPC_High",2) - ROUND("^GSPC_Low",2) AS price_diff
      FROM stocks_data
      ORDER BY price_diff DESC
      LIMIT 5;""",

      "stocks(19).Get monthly average closing price for each ticker":
      """SELECT strftime('%Y-%m', Date) AS month,
      ROUND(AVG("^GSPC_Close"),2) AS SP500,
      ROUND(AVG("^NSEI_Close"),2) AS NIFTY,
      ROUND(AVG("^IXIC_Close"),2) AS NASDAQ
      FROM stocks_data
      GROUP BY month
      ORDER BY month;""",

      "stocks(20).Get average trading volume of NSEI in 2024":
      """SELECT AVG("^NSEI_Volume") AS avg_volume
      FROM stocks_data
      WHERE Date BETWEEN '2024-01-01' AND '2024-12-31';""",

      "joins(21).Compare Bitcoin vs Oil average price in 2025":
      """SELECT ROUND(AVG(c.price_inr),2) AS bitcoin_avg,
      ROUND(AVG(o.price),2) AS oil_avg
      FROM coins_data c
      INNER JOIN oil_data o
      ON c.date = o.date
      WHERE c.date BETWEEN '2025-01-01' AND '2025-12-31';""",

      "joins(22).Check if Bitcoin moves with S&P 500 (correlation idea)":
      """SELECT ROUND(AVG(c.price_inr),2) AS bitcoin_avg,
      ROUND(AVG(s."^GSPC_Close"),2) AS sp500_avg
      FROM coins_data c
      INNER JOIN stocks_data s
      ON c.date = s.Date
      WHERE c.coin_id='bitcoin';""",

      "joins(23).Compare Ethereum and NASDAQ daily prices for 2025":
      """SELECT ROUND(AVG(c.price_inr),2) AS ethereum_avg,
      ROUND(AVG(s."^IXIC_Close"),2) AS nasdaq_avg
      FROM coins_data c
      INNER JOIN stocks_data s
      ON c.date = s.Date
      WHERE c.coin_id='ethereum'
      AND c.date BETWEEN '2025-01-01' AND '2025-12-31';""",

      "joins(24).Find days when oil price spiked and compare with Bitcoin price change":
      """SELECT
      o.Date,
      ROUND(o.Price, 2) AS oil_price,
      ROUND(c.price_inr, 2) AS bitcoin_price
      FROM oil_data o
      JOIN coins_data c
      ON o.Date = c.date
      WHERE c.coin_id = 'bitcoin'
      ORDER BY o.Date;""",

      "joins(25).Compare top 3 coins daily price trend vs Nifty (^NSEI)":
      """SELECT c.date,
      ROUND(c.price_inr,2) AS bitcoin,
      ROUND(c.price_inr,2) AS ethereum,
      ROUND(c.price_inr,2) AS dogecoin,
      ROUND(s."^NSEI_Close",2) AS nifty
      FROM coins_data c
      INNER JOIN stocks_data s
      ON c.date = s.Date
      WHERE c.date BETWEEN '2025-01-01' AND '2025-12-31'
      ORDER BY c.date;""",

      "joins(26).Compare stock prices (^GSPC) with crude oil prices on the same dates":
      """SELECT s."^GSPC_Close", o.price
      FROM stocks_data s
      INNER JOIN oil_data o
      ON s.Date = o.Date
      WHERE s.Date BETWEEN '2025-01-01' AND '2025-12-31';""",

      "joins(27).Correlate Bitcoin closing price with crude oil closing price (same date)":
      """SELECT c.date, ROUND(AVG(c.price_inr),2) AS bitcoin_avg,
      ROUND(AVG(o.price),2) AS oil_avg
      FROM coins_data c
      INNER JOIN oil_data o
      ON c.date = o.Date
      WHERE c.coin_id='bitcoin' group by c.date limit 1;""",

      "joins(28).Compare NASDAQ (^IXIC) with Ethereum price trends":
      """SELECT c.date, ROUND(AVG(c.price_inr),2) AS ethereum_avg,
      ROUND(AVG(s."^IXIC_Close"),2) AS nasdaq_avg
      FROM coins_data c
      INNER JOIN stocks_data s
      ON c.date = s.Date
      WHERE c.coin_id = 'ethereum' GROUP BY c.date;""",

      "joins(29).Join top 3 crypto coins with stock indices for 2025":
      """SELECT c.date,
      ROUND(c.price_inr,2) AS bitcoin,
      ROUND(c.price_inr,2) AS ethereum,
      ROUND(c.price_inr,2) AS dogecoin,
      ROUND(s."^GSPC_Close",2) AS sp500,
      ROUND(s."^NSEI_Close",2) AS nifty,
      ROUND(s."^IXIC_Close",2) AS nasdaq
      FROM coins_data c
      INNER JOIN stocks_data s
      ON c.date = s.Date
      WHERE c.date BETWEEN '2025-01-01' AND '2025-12-31'
      ORDER BY c.date;""",

      "joins(30).Multi-join: stock prices, oil prices, and Bitcoin prices for daily comparison":
      """SELECT s.Date,
      ROUND(s."^GSPC_Close",2) AS sp500,
      ROUND(s."^NSEI_Close",2) AS nifty,
      ROUND(s."^IXIC_Close",2) AS nasdaq,
      ROUND(o.price,2) AS oil_price,
      ROUND(c.price_inr,2) AS bitcoin_price
      FROM stocks_data s
      INNER JOIN oil_data o
      ON s.Date = o.Date
      INNER JOIN coins_data c
      ON s.Date = c.date
      WHERE c.coin_id = 'bitcoin'
      ORDER BY s.Date;"""

  }
  option = st.selectbox("Select a query to execute:", list(query_option.keys()))

  if st.button("Run Query"):
        try:
            result_df = pd.read_sql_query(query_option[option], conn)
            st.dataframe(result_df, use_container_width=True)
        except Exception as e:
            st.error(f"Query failed: {e}")

elif selected == "Top 03 Crypto Analysis":
    st.title('Welcome to Top 03 Crypto Analysis')

    # --- Get list of available coins (Top 3) ---
    coins_df = pd.read_sql("SELECT DISTINCT coin_id FROM coins_data", con=conn)
    coin_list = coins_df['coin_id'].tolist()

    selected_coin = st.selectbox("Select a Cryptocurrency", coin_list)

    # --- Get min/max date for that coin, to bound the date picker ---
    date_range = pd.read_sql(
        "SELECT MIN(date) as min_date, MAX(date) as max_date FROM coins_data WHERE coin_id = ?",
        con=conn,
        params=(selected_coin,)
    )
    min_date = pd.to_datetime(date_range['min_date'][0])
    max_date = pd.to_datetime(date_range['max_date'][0])

    # --- Date range picker (safely handled) ---
    date_selection = st.date_input(
        "Select Date Range",
        value=(min_date, max_date),
        min_value=min_date,
        max_value=max_date
    )

    # Guard against the case where only one date has been picked so far
    if not isinstance(date_selection, tuple) or len(date_selection) != 2:
        st.warning("Please select both a start and end date.")
        st.stop()

    start_date, end_date = date_selection

    # Push end date to end-of-day so BETWEEN includes the full last day
    start_date_str = str(start_date) + " 00:00:00"
    end_date_str = str(end_date) + " 23:59:59"

    # --- Query filtered data ---
    query = """
    SELECT date, price_inr
    FROM coins_data
    WHERE coin_id = ?
    AND date BETWEEN ? AND ?
    ORDER BY date;
    """
    df = pd.read_sql(query, con=conn, params=(selected_coin, start_date_str, end_date_str))

    if df.empty:
        st.warning("No data available for the selected coin and date range.")
    else:
        df['date'] = pd.to_datetime(df['date'])

        # --- Line chart (optional per spec, but included) ---
        st.subheader(f"{selected_coin.capitalize()} Price Trend")
        st.line_chart(df.set_index('date')['price_inr'])

        # --- Data table ---
        st.subheader("Daily Price Table")
        st.dataframe(
            df.rename(columns={'date': 'Date', 'price_inr': 'Price (INR)'}),
            use_container_width=True
        )

Delete exist streamlit

In [ ]:
!streamlit run /content/crossmarketanalysis.py &>/content/logs.txt &

In [ ]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"